# Notebook 07

# Engine 01

Document Ingestion Engine

---

## Goal

Build a reusable document ingestion engine.

---

## Inputs

Data/raw/

---

## Outputs

Data/processed/

metadata/

Document Object

---

## Engine Produced

IngestionEngine

---

## Used By

ChunkEngine

EmbeddingEngine

Retriever

Generation Engine

In [20]:
!pip install -q \
pymupdf \
python-docx \
sentence-transformers \
chromadb \
transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/6

In [21]:
import os
import re
import json
import uuid

from pathlib import Path

import fitz

from google.colab import drive

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os

ROOT = "/content/drive/MyDrive/MicroBrain"

print(os.path.exists(ROOT))

True


In [22]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/MicroBrain")

DATA = ROOT / "Data"

RAW = DATA / "raw"

PROCESSED = DATA / "processed"

METADATA = DATA / "metadata"

CHUNKS = DATA / "chunks"

EMBEDDINGS = DATA / "embeddings"

VECTORDB = DATA / "vectordb"

ENGINES = ROOT / "Engines"

MODELS = ROOT / "Models"

OUTPUTS = ROOT / "Outputs"

In [23]:
paths = [
    RAW,
    PROCESSED,
    METADATA,
    CHUNKS,
    EMBEDDINGS,
    VECTORDB,
    ENGINES,
    MODELS,
    OUTPUTS,
]

for path in paths:
    print(
        f"{path.name:<12}",
        "✓" if path.exists() else "✗"
    )

raw          ✓
processed    ✓
metadata     ✓
chunks       ✓
embeddings   ✓
vectordb     ✓
Engines      ✓
Models       ✓
Outputs      ✓


In [8]:
from dataclasses import dataclass, field

from typing import List, Dict, Optional

# Phase 2 - Create Standard Document Object

Instead of passing plain text between engines, we create a
structured document object.

This object contains:

- Document ID
- File Name
- Number of Pages
- Original Length
- Clean Text
- Metadata

Future engines will add:

- Chunks
- Embeddings
- Retrieval Scores
- Generated Answers

Everything revolves around this object.

In [18]:
from pathlib import Path

pdf_files = sorted(RAW_DATA.glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF(s)\n")

for pdf in pdf_files:
    print(pdf.name)

Found 1 PDF(s)

Transformer_Math_and_RAG_Explained.pdf


In [24]:
pdf_path = pdf_files[0]

pdf_document = fitz.open(pdf_path)

print(pdf_document)

print()

print("Pages :", len(pdf_document))

Document('/content/drive/MyDrive/MicroBrain/Data/raw/Transformer_Math_and_RAG_Explained.pdf')

Pages : 6


## Step 1 - Extract Text

Read every page in the PDF and combine the text into one
continuous string.

Output

text (raw extracted text)

In [25]:
text = ""

for page in pdf_document:
    text += page.get_text()

print("Characters :", len(text))

Characters : 9838


## Step 2 - Clean Text

Normalize whitespace so downstream engines always receive
consistent text.

Output

clean_text

In [26]:
import re

clean_text = re.sub(r"\s+", " ", text)

clean_text = clean_text.strip()

print("Characters :", len(clean_text))

Characters : 9837


## Step 3 - Create Standard Document

Instead of passing text between notebooks,
create a structured document object.

Every future engine will receive this object.

In [32]:
import uuid

document_id = str(uuid.uuid4())

document = {

    "id": document_id,

    "source": {

        "filename": pdf_path.name,

        "filepath": str(pdf_path),

        "type": pdf_path.suffix.replace(".", "").lower()

    },

    "metadata": {

        "pages": len(pdf_document),

        "characters": len(clean_text),

        "language": "english",

        "processed": True

    },

    "content": {

        "text": clean_text

    }

}

## Step 4 - Save Processed Document

Save the standardized document object.

Future engines will load this file directly instead
of rebuilding it.

In [36]:
import json

document_path = PROCESSED / f"{document_id}.json"

with open(document_path, "w", encoding="utf-8") as file:

    json.dump(
        document,
        file,
        indent=4
    )

print(document_path)

/content/drive/MyDrive/MicroBrain/Data/processed/a60be88b-ec91-429e-8ab4-c1bb43cdf6e6.json


# Step 5 - Document Registry

Maintain a registry of every ingested document.

The registry is the source of truth for all processed
documents inside the system.

In [37]:
registry_path = METADATA / "documents.json"

In [38]:
import json

if registry_path.exists():

    with open(registry_path, "r") as file:

        registry = json.load(file)

else:

    registry = []

In [39]:
registry.append({

    "id": document["id"],

    "filename": document["source"]["filename"],

    "processed_document": document_path.name

})

In [40]:
with open(registry_path, "w") as file:

    json.dump(
        registry,
        file,
        indent=4
    )

print(registry_path)

/content/drive/MyDrive/MicroBrain/Data/metadata/documents.json


In [41]:
already_processed = False

for doc in registry:

    if doc["filename"] == pdf_path.name:

        already_processed = True

        break

In [42]:
if already_processed:

    print("Already processed.")

else:

    print("Continue processing.")

Already processed.


# Engine Outputs

Input

Data/raw/*.pdf

Outputs

Data/processed/<document_id>.json

Data/metadata/documents.json

Purpose

Convert raw enterprise documents into standardized
Document Objects that can be consumed by downstream
engines.

Next Engine

Chunking Engine